In [8]:
import pandas as pd
import os
import datetime

# Step 1: Automatically detect the CSV file that starts with "teal_iq" and Excel file that contains "VM Analysis" in the current directory
folder_path = "."  # Current folder
file_name = None
vm_file_name = None

# Search for a file that starts with "teal_iq" and ends with ".csv", and the Excel file for "VM Analysis"
for f in os.listdir(folder_path):
    if f.startswith("matched") and f.endswith(".csv"):
        file_name = f
    if "VM Analysis" in f and f.endswith(".xlsx"):
        vm_file_name = f

if not file_name:
    raise FileNotFoundError("No CSV file starting with 'teal_iq' found in the folder.")
if not vm_file_name:
    raise FileNotFoundError("No Excel file containing 'VM Analysis' found in the folder.")

# Read the input CSV file with low_memory=False to handle mixed data types
input_file = os.path.join(folder_path, file_name)
df = pd.read_csv(input_file, low_memory=False)

# Read the "Spend Summary" tab from the VM Analysis Excel file
vm_analysis_file = os.path.join(folder_path, vm_file_name)
vm_df = pd.read_excel(vm_analysis_file, sheet_name='Spend Summary')

# Define the enriched fields we need to extract for both tabs
enriched_fields = ['company_name', 'complete_address', 'description', 'web_domain', 'contact_name', 
                   'contact_title', 'contact_email', 'contact_phone_number', 'duns_number', 'naics',
                   'year_founded', 'cage_code', 'lei', 'sam_uei', 'employee_count', 'phone_number',
                   'annual_revenue_number', 'gross_profit', 'local_exchange_symbol', 'net_income',
                   'operating_income', 'total_assets', 'total_liabilities', 'sic']

# Helper function to concatenate merged values
def concatenate_fields(filtered_df, enriched_field):
    enriched_field_df = filtered_df[filtered_df['enriched_field'] == enriched_field]
    merged_values = enriched_field_df[['merged_value_1', 'merged_value_2', 'merged_value_3', 
                                       'merged_value_4', 'merged_value_5', 'merged_value_6', 
                                       'merged_value_7', 'merged_value_8', 'merged_value_9', 
                                       'merged_value_10', 'merged_value_11', 'merged_value_12', 
                                       'merged_value_13', 'merged_value_14', 'merged_value_15', 
                                       'merged_value_16', 'merged_value_17', 'merged_value_18', 
                                       'merged_value_19', 'merged_value_20']]
    concatenated_values = merged_values.apply(lambda x: '|'.join(x.dropna().astype(str)), axis=1).tolist()
    return '|'.join(concatenated_values) if concatenated_values else ''

# Merge the "VM Analysis" file with the main data based on 'internal_supplier_id_or_vendor_number'
vm_analysis_spend_map = vm_df.set_index('internal supplier id')['aggregated spend']

# Step 2: Firm vs Establishment Tab Logic - Deduplicate on three columns
firm_vs_establishment_data = []
deduplicated_df_fve = df.drop_duplicates(subset=['internal_supplier_id_or_vendor_number', 'tealbook_id', 'entity_type'])

# Process each deduplicated row for Firm vs Establishment
for _, row in deduplicated_df_fve.iterrows():
    internal_supplier_id_or_vendor_number = row['internal_supplier_id_or_vendor_number']
    tealbook_id = row['tealbook_id']
    entity_type = row['entity_type']
    
    # Filter the original dataframe based on these three values
    filtered_df = df[(df['internal_supplier_id_or_vendor_number'] == internal_supplier_id_or_vendor_number) & 
                     (df['tealbook_id'] == tealbook_id) & 
                     (df['entity_type'] == entity_type)]
    
    # Prepare a dictionary for the row
    row_data = {
        'internal_supplier_id_or_vendor_number': internal_supplier_id_or_vendor_number,
        'tealbook_id': tealbook_id,
        'entity_type': entity_type
    }
    
    # Concatenate the enriched fields
    for enriched_field in enriched_fields:
        row_data[enriched_field] = concatenate_fields(filtered_df, enriched_field)
    
    # Handle blank 'company_name' and 'complete_address' by matching 'internal_supplier_id_or_vendor_number'
    if not row_data['company_name']:  # If 'company_name' is blank
        company_name_row = df[(df['internal_supplier_id_or_vendor_number'] == internal_supplier_id_or_vendor_number) &
                              (df['enriched_field'] == 'company_name')]
        if not company_name_row.empty:
            row_data['company_name'] = company_name_row['merged_value_1'].values[0]  # Populate from 'merged_value_1'
    
    if not row_data['complete_address']:  # If 'complete_address' is blank
        complete_address_row = df[(df['internal_supplier_id_or_vendor_number'] == internal_supplier_id_or_vendor_number) &
                                  (df['enriched_field'] == 'complete_address')]
        if not complete_address_row.empty:
            row_data['complete_address'] = complete_address_row['merged_value_1'].values[0]  # Populate from 'merged_value_1'
    
    # Add the "aggregated spend" column from VM Analysis file
    row_data['aggregated spend'] = vm_analysis_spend_map.get(internal_supplier_id_or_vendor_number, None)
    
    # Append the row data to the output list
    firm_vs_establishment_data.append(row_data)

# Convert the processed data for Firm vs Establishment into a dataframe
firm_vs_establishment_df = pd.DataFrame(firm_vs_establishment_data)

# Step 3: Consolidated Rows Tab Logic - Deduplicate on 'internal_supplier_id_or_vendor_number' only
consolidated_rows_data = []
deduplicated_df_fve = df.drop_duplicates(subset=['internal_supplier_id_or_vendor_number'])

# Process each deduplicated row for Consolidated Rows
for _, row in deduplicated_df_fve.iterrows():
    internal_supplier_id_or_vendor_number = row['internal_supplier_id_or_vendor_number']
    
    # Filter the original dataframe based on 'internal_supplier_id_or_vendor_number' only
    filtered_df = df[df['internal_supplier_id_or_vendor_number'] == internal_supplier_id_or_vendor_number]
    
    # Prepare a dictionary for the row
    row_data = {
        'internal_supplier_id_or_vendor_number': internal_supplier_id_or_vendor_number
    }
    
    # Concatenate the enriched fields
    for enriched_field in enriched_fields:
        row_data[enriched_field] = concatenate_fields(filtered_df, enriched_field)
    
    # Add the "aggregated spend" column from VM Analysis file
    row_data['aggregated spend'] = vm_analysis_spend_map.get(internal_supplier_id_or_vendor_number, None)
    
    # Append the row data to the output list
    consolidated_rows_data.append(row_data)

# Convert the processed data for Consolidated Rows into a dataframe
consolidated_rows_df = pd.DataFrame(consolidated_rows_data)

# Calculate the count of unique internal IDs and the total aggregated spend
unique_internal_ids_count = consolidated_rows_df['internal_supplier_id_or_vendor_number'].nunique()
unique_internal_ids_spend = consolidated_rows_df['aggregated spend'].sum()

# Calculate "Spend" values for each category
spend_values = [
    unique_internal_ids_spend,  # Unique Internal IDs spend
    consolidated_rows_df.loc[consolidated_rows_df['company_name'].notna() & (consolidated_rows_df['company_name'].str.strip() != ''), 'aggregated spend'].sum(),
    consolidated_rows_df.loc[consolidated_rows_df['complete_address'].notna() & (consolidated_rows_df['complete_address'].str.strip() != ''), 'aggregated spend'].sum(),
    consolidated_rows_df.loc[consolidated_rows_df['description'].notna() & (consolidated_rows_df['description'].str.strip() != ''), 'aggregated spend'].sum(),
    consolidated_rows_df.loc[consolidated_rows_df['web_domain'].notna() & (consolidated_rows_df['web_domain'].str.strip() != ''), 'aggregated spend'].sum(),
    consolidated_rows_df.loc[consolidated_rows_df['contact_name'].notna() & (consolidated_rows_df['contact_name'].str.strip() != ''), 'aggregated spend'].sum(),
    consolidated_rows_df.loc[consolidated_rows_df['contact_title'].notna() & (consolidated_rows_df['contact_title'].str.strip() != ''), 'aggregated spend'].sum(),
    consolidated_rows_df.loc[consolidated_rows_df['contact_email'].notna() & (consolidated_rows_df['contact_email'].str.strip() != ''), 'aggregated spend'].sum(),
    consolidated_rows_df.loc[consolidated_rows_df['contact_phone_number'].notna() & (consolidated_rows_df['contact_phone_number'].str.strip() != ''), 'aggregated spend'].sum(),
    consolidated_rows_df.loc[consolidated_rows_df['duns_number'].notna() & (consolidated_rows_df['duns_number'].str.strip() != ''), 'aggregated spend'].sum(),
    consolidated_rows_df.loc[consolidated_rows_df['naics'].notna() & (consolidated_rows_df['naics'].str.strip() != ''), 'aggregated spend'].sum(),
    consolidated_rows_df.loc[consolidated_rows_df['year_founded'].notna() & (consolidated_rows_df['year_founded'].str.strip() != ''), 'aggregated spend'].sum(),
    consolidated_rows_df.loc[consolidated_rows_df['cage_code'].notna() & (consolidated_rows_df['cage_code'].str.strip() != ''), 'aggregated spend'].sum(),
    consolidated_rows_df.loc[consolidated_rows_df['lei'].notna() & (consolidated_rows_df['lei'].str.strip() != ''), 'aggregated spend'].sum(),
    consolidated_rows_df.loc[consolidated_rows_df['sam_uei'].notna() & (consolidated_rows_df['sam_uei'].str.strip() != ''), 'aggregated spend'].sum(),
    consolidated_rows_df.loc[consolidated_rows_df['employee_count'].notna() & (consolidated_rows_df['employee_count'].str.strip() != ''), 'aggregated spend'].sum(),
    consolidated_rows_df.loc[consolidated_rows_df['phone_number'].notna() & (consolidated_rows_df['phone_number'].str.strip() != ''), 'aggregated spend'].sum(),
    consolidated_rows_df.loc[consolidated_rows_df['annual_revenue_number'].notna() & (consolidated_rows_df['annual_revenue_number'].str.strip() != ''), 'aggregated spend'].sum(),
    consolidated_rows_df.loc[consolidated_rows_df['local_exchange_symbol'].notna() & (consolidated_rows_df['local_exchange_symbol'].str.strip() != ''), 'aggregated spend'].sum(),
    consolidated_rows_df.loc[consolidated_rows_df['sic'].notna() & (consolidated_rows_df['sic'].str.strip() != ''), 'aggregated spend'].sum(),
    "n/a"
]

# Define enriched fields that match the length of categories, excluding Unique IDs and Average Attributes
enriched_fields_for_summary = enriched_fields[:19]

# Reconstruct summary_data with explicit checks on each array's length
summary_data = {
    "Category": [
        "Unique Internal IDs", "Company Name", "Complete Address", "Description", "Web Domain",
        "Contact Name", "Contact Title", "Contact Email", "Contact Phone", "DUNS", "NAICS",
        "Year Founded", "Cage Code", "LEI", "SAM UEI", "Employee Count", "Phone Number",
        "Annual Revenue", "Local Exchange Symbol", "SIC", "Average Attributes"
    ],
    # Counts of non-null values for each field
    "Count": [
        unique_internal_ids_count,
        *[
            consolidated_rows_df[field].replace('', None).notna().sum()
            for field in enriched_fields_for_summary
        ],
        round(consolidated_rows_df.iloc[:, 3:25].replace('', None).notna().sum(axis=1).mean(), 2)
    ],
    # Percentage calculations based on Count
    "Percentage": [
        (count / unique_internal_ids_count) if count != 'n/a' else ''
        for count in [
            unique_internal_ids_count,
            consolidated_rows_df['company_name'].replace('', None).notna().sum(),
            consolidated_rows_df['complete_address'].replace('', None).notna().sum(),
            consolidated_rows_df['description'].replace('', None).notna().sum(),
            consolidated_rows_df['web_domain'].replace('', None).notna().sum(),
            consolidated_rows_df['contact_name'].replace('', None).notna().sum(),
            consolidated_rows_df['contact_title'].replace('', None).notna().sum(),
            consolidated_rows_df['contact_email'].replace('', None).notna().sum(),
            consolidated_rows_df['contact_phone_number'].replace('', None).notna().sum(),
            consolidated_rows_df['duns_number'].replace('', None).notna().sum(),
            consolidated_rows_df['naics'].replace('', None).notna().sum(),
            consolidated_rows_df['year_founded'].replace('', None).notna().sum(),
            consolidated_rows_df['cage_code'].replace('', None).notna().sum(),
            consolidated_rows_df['lei'].replace('', None).notna().sum(),
            consolidated_rows_df['sam_uei'].replace('', None).notna().sum(),
            consolidated_rows_df['employee_count'].replace('', None).notna().sum(),
            consolidated_rows_df['phone_number'].replace('', None).notna().sum(),
            consolidated_rows_df['annual_revenue_number'].replace('', None).notna().sum(),
            consolidated_rows_df['local_exchange_symbol'].replace('', None).notna().sum(),
            consolidated_rows_df['sic'].replace('', None).notna().sum(),
            "n/a"
        ]
    ],
    # Spend values based on aggregated spend for each field
    "Spend": [
        unique_internal_ids_spend,
        *[
            consolidated_rows_df.loc[consolidated_rows_df[field].notna() & (consolidated_rows_df[field].str.strip() != ''), 'aggregated spend'].sum()
            for field in enriched_fields_for_summary
        ],
        "n/a"
    ],
    # Spend Percentage calculations
    "Spend Percentage": [
        (spend / unique_internal_ids_spend) if unique_internal_ids_spend != 0 and spend != "n/a" else ''
        for spend in [
            unique_internal_ids_spend,
            *[
                consolidated_rows_df.loc[consolidated_rows_df[field].notna() & (consolidated_rows_df[field].str.strip() != ''), 'aggregated spend'].sum()
                for field in enriched_fields_for_summary
            ],
            "n/a"
        ]
    ]
}

# Create the final DataFrame for Consolidated Summary
summary_df = pd.DataFrame(summary_data)

# Export to Excel (omitting formatting for brevity)
today_date = datetime.datetime.today().strftime('%Y-%m-%d')
output_file = f"Step 6_SDP Matched Consolidated - {today_date}.xlsx"

# Step 1: Read "Vendor Master" tab from the VM Analysis file
vendor_master_df = pd.read_excel(vm_analysis_file, sheet_name="Vendor Master")

# Ensure required columns exist in the "Vendor Master" tab
required_columns = ['internal supplier id', 'code']
missing_columns = [col for col in required_columns if col not in vendor_master_df.columns]
if missing_columns:
    raise KeyError(f"Missing columns in 'Vendor Master' tab: {missing_columns}")

# Prepare the Firm vs Establishment and Consolidated Rows DataFrames
firm_vs_establishment_df = pd.DataFrame(firm_vs_establishment_data)
consolidated_rows_df = pd.DataFrame(consolidated_rows_data)

# Normalize and clean IDs for consistent mapping
vendor_master_df['internal supplier id'] = vendor_master_df['internal supplier id'].astype(str).str.strip()
firm_vs_establishment_df['internal_supplier_id_or_vendor_number'] = firm_vs_establishment_df['internal_supplier_id_or_vendor_number'].astype(str).str.strip()
consolidated_rows_df['internal_supplier_id_or_vendor_number'] = consolidated_rows_df['internal_supplier_id_or_vendor_number'].astype(str).str.strip()

# Check for matched IDs
matched_ids = set(firm_vs_establishment_df['internal_supplier_id_or_vendor_number']) & set(vendor_master_df['internal supplier id'])
print(f"Matched IDs: {len(matched_ids)} out of {len(firm_vs_establishment_df['internal_supplier_id_or_vendor_number'].unique())}")

# Explicitly join to validate mapping for Firm vs Establishment
merged_firm_df = firm_vs_establishment_df.merge(
    vendor_master_df[['internal supplier id', 'code']],
    left_on='internal_supplier_id_or_vendor_number',
    right_on='internal supplier id',
    how='left'
)
merged_firm_df['Person?'] = merged_firm_df['code'].fillna("").apply(lambda x: "YES" if str(x).strip() == "P" else "")
firm_vs_establishment_df = merged_firm_df.drop(columns=['internal supplier id', 'code'])

# Explicitly join to validate mapping for Consolidated Rows
merged_consolidated_df = consolidated_rows_df.merge(
    vendor_master_df[['internal supplier id', 'code']],
    left_on='internal_supplier_id_or_vendor_number',
    right_on='internal supplier id',
    how='left'
)
merged_consolidated_df['Person?'] = merged_consolidated_df['code'].fillna("").apply(lambda x: "YES" if str(x).strip() == "P" else "")
consolidated_rows_df = merged_consolidated_df.drop(columns=['internal supplier id', 'code'])

# Reorder columns so "Person?" appears in column A
firm_vs_establishment_df = firm_vs_establishment_df[['Person?'] + [col for col in firm_vs_establishment_df.columns if col != 'Person?']]
consolidated_rows_df = consolidated_rows_df[['Person?'] + [col for col in consolidated_rows_df.columns if col != 'Person?']]

# Deduplicate Consolidated Rows based on internal_supplier_id_or_vendor_number
consolidated_rows_df = consolidated_rows_df.drop_duplicates(subset=['internal_supplier_id_or_vendor_number'], keep='first')

# Step 5: Save to Excel with Formatting
with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
    # Write the Firm vs Establishment sheet
    firm_vs_establishment_df.to_excel(writer, sheet_name="Firm vs Establishment", index=False)
    worksheet_firm = writer.sheets["Firm vs Establishment"]
    worksheet_firm.set_column('A:A', 10)  # Adjust width for "Person?"

    # Write the Consolidated Rows sheet
    consolidated_rows_df.to_excel(writer, sheet_name="Consolidated Rows", index=False)
    worksheet_consolidated = writer.sheets["Consolidated Rows"]
    worksheet_consolidated.set_column('A:A', 10)  # Adjust width for "Person?"

    # Additional formatting and conditional formatting can go here

with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
    workbook = writer.book
    
    # Define formats for headers and rows
    header_format_teal = workbook.add_format({'bold': True, 'bg_color': '#008080', 'font_color': 'white'})
    header_format_orange = workbook.add_format({'bold': True, 'bg_color': '#FFA500', 'font_color': 'white'})
    firm_row_format_teal = workbook.add_format({'bg_color': '#CCFFFF'})
    establishment_row_format_orange = workbook.add_format({'bg_color': '#FFCC99'})
    percentage_format = workbook.add_format({'num_format': '0.00%'})  # Percentage format for Percentage column
    number_format = workbook.add_format({'num_format': '#,##0'})  # Number formatting for Spend
    
    # Write the Firm vs Establishment sheet
    firm_vs_establishment_df = firm_vs_establishment_df.sort_values(by=['internal_supplier_id_or_vendor_number'])
    firm_vs_establishment_df.to_excel(writer, sheet_name="Firm vs Establishment", index=False)
    worksheet_firm = writer.sheets["Firm vs Establishment"]
    
    # Set column widths and apply header format
    worksheet_firm.set_column('A:A', 18)  # internal_supplier_id_or_vendor_number
    worksheet_firm.set_column('B:B', 20, number_format)  # aggregated spend
    worksheet_firm.set_column('C:C', 35)  # tealbook_id
    worksheet_firm.set_column('D:D', 15)  # entity_type
    worksheet_firm.set_column('E:F', 35)  # company_name and complete_address
    worksheet_firm.set_column('G:Z', 20)  # Remaining columns
    
    # Apply teal header format
    for col_num, value in enumerate(firm_vs_establishment_df.columns.values):
        worksheet_firm.write(0, col_num, value, header_format_teal)
    
    # Apply row formatting based on sorted entity_type
    for row_num in range(1, len(firm_vs_establishment_df) + 1):
        entity_type = firm_vs_establishment_df.iloc[row_num - 1]['entity_type']
        if entity_type == "FIRM":
            worksheet_firm.set_row(row_num, None, firm_row_format_teal)
        elif entity_type == "ESTABLISHMENT":
            worksheet_firm.set_row(row_num, None, establishment_row_format_orange)
    
    # Write the Consolidated Rows sheet
    consolidated_rows_df = consolidated_rows_df.sort_values(by=['internal_supplier_id_or_vendor_number'])
    consolidated_rows_df.to_excel(writer, sheet_name="Consolidated Rows", index=False)
    worksheet_consolidated = writer.sheets["Consolidated Rows"]
    
    # Set column widths and apply header format
    worksheet_consolidated.set_column('A:A', 18)  # internal_supplier_id_or_vendor_number
    worksheet_consolidated.set_column('B:B', 20, number_format)  # aggregated spend
    worksheet_consolidated.set_column('C:D', 35)  # company_name, complete_address
    worksheet_consolidated.set_column('E:Z', 20)  # Remaining columns
    
    # Apply orange header format
    for col_num, value in enumerate(consolidated_rows_df.columns.values):
        worksheet_consolidated.write(0, col_num, value, header_format_orange)

    # Write the Consolidated Summary sheet
    summary_df.to_excel(writer, sheet_name="Consolidated Summary", index=False)
    worksheet_summary = writer.sheets["Consolidated Summary"]
    
    # Set column widths for the Consolidated Summary tab
    worksheet_summary.set_column('A:A', 25)  # Category
    worksheet_summary.set_column('B:B', 15)  # Count
    worksheet_summary.set_column('C:C', 15, percentage_format)  # Percentage
    worksheet_summary.set_column('D:D', 20, number_format)  # Spend
    worksheet_summary.set_column('E:E', 15, percentage_format)  # Spend Percentage
    
    # Apply teal header format to Consolidated Summary
    for col_num, value in enumerate(summary_df.columns.values):
        worksheet_summary.write(0, col_num, value, header_format_teal)

    # Apply conditional formatting (orange for max, white for min) for Count, Percentage, Spend, and Spend Percentage columns
    worksheet_summary.conditional_format('B2:B{}'.format(len(summary_df) + 1), {
        'type': '2_color_scale',
        'min_color': "#FFFFFF",
        'max_color': "#FFA500"
    })
    worksheet_summary.conditional_format('C2:C{}'.format(len(summary_df) + 1), {
        'type': '2_color_scale',
        'min_color': "#FFFFFF",
        'max_color': "#FFA500"
    })
    worksheet_summary.conditional_format('D2:D{}'.format(len(summary_df) + 1), {
        'type': '2_color_scale',
        'min_color': "#FFFFFF",
        'max_color': "#FFA500"
    })
    worksheet_summary.conditional_format('E2:E{}'.format(len(summary_df) + 1), {
        'type': '2_color_scale',
        'min_color': "#FFFFFF",
        'max_color': "#FFA500"
    })

print(vendor_master_df.head())
print(vendor_master_df.columns)
print(firm_vs_establishment_df['internal_supplier_id_or_vendor_number'].head())
print(vendor_master_df['internal supplier id'].head())
print(firm_vs_establishment_df[['internal_supplier_id_or_vendor_number', 'Person?']].head())
print(firm_vs_establishment_df.columns)
print(consolidated_rows_df.columns)

print(f"Excel file '{output_file}' created successfully!")


Matched IDs: 3205 out of 3205
  code        spendcountry sanctioned country internal supplier id  \
0  NaN       United States                NaN               101264   
1  NaN               Spain                NaN               106338   
2  NaN       United States                NaN               107160   
3  NaN       United States                NaN               107271   
4  NaN  Korea, Republic of                NaN               112562   

                          company name  \
0                     Pitney Bowes Inc   
1       SegurCaixa Adeslas SA de Segur   
2  American Arbitration Association, I   
3                   SAS Institute Inc.   
4                   Expertnet Co., Ltd   

                               complete full address  contact email  \
0         27 Waterview Drive, Shelton, CT, 06484, US            NaN   
1    Juan Gris 20-26 20-26, Barcelona, 28, 08014, ES            NaN   
2     120 Broadway, 21st FL, New York, NY, 10271, US            NaN   
3         PO